# 🫒 OliveVision — Training Notebook
## Real-Time Olive Detection & Counting
### Architecture: RT-DETR-L with CBAM + BiFPN + IoU-Aware Head

---

**Table of Contents**
1. [Environment Setup](#1-environment-setup)
2. [Configuration](#2-configuration)
3. [Dataset Preparation](#3-dataset-preparation)
4. [Exploratory Data Analysis](#4-exploratory-data-analysis)
5. [Model Architecture Overview](#5-model-architecture-overview)
6. [Training](#6-training)
7. [Evaluation](#7-evaluation)
8. [Export & Deployment](#8-export--deployment)
9. [Quick Inference Test](#9-quick-inference-test)

---
**Why RT-DETR?**
- End-to-end detection without NMS (faster, cleaner)
- State-of-the-art accuracy on dense/small object detection
- Real-time capable at 640×640 on a single GPU
- COCO pretrained backbone = excellent transfer learning for olives


---
## 1 — Environment Setup

In [1]:
# ── Install dependencies ─────────────────────────────────────────────────
import subprocess, sys

def pip_install(*packages):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages], check=True)

pip_install(
    'torch', 'torchvision',
    'ultralytics>=8.2.0',
    'albumentations>=1.4.0',
    'pycocotools',
    'scipy',
    'pyyaml',
    'tensorboard',
    'tqdm',
)
print('✅ All packages installed')

✅ All packages installed


In [2]:
# ── Core imports ─────────────────────────────────────────────────────────
import os, sys, yaml, json, time, math, random, shutil, warnings
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from collections import defaultdict
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from ultralytics import RTDETR

warnings.filterwarnings('ignore')

# ── Add project root to path ─────────────────────────────────────────────
PROJECT_ROOT = Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'📁 Project root: {PROJECT_ROOT}')
print(f'🐍 Python: {sys.version.split()[0]}')
print(f'🔥 PyTorch: {torch.__version__}')
print(f'🖥️  CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

📁 Project root: /content
🐍 Python: 3.12.12
🔥 PyTorch: 2.9.0+cu126
🖥️  CUDA available: True
   GPU: Tesla T4
   VRAM: 15.6 GB


---
## 2 — Configuration

In [3]:
# ── Load config ───────────────────────────────────────────────────────
from utils.utils import load_config, set_seed, get_device

# Use PROJECT_ROOT to construct the correct absolute path
CONFIG_PATH = PROJECT_ROOT / 'config' / 'config.yaml'
print(f'🔍 Looking for config at: {CONFIG_PATH}')
print(f'   File exists: {CONFIG_PATH.exists()}')

if not CONFIG_PATH.exists():
    print(f'❌ Config NOT found! Checked: {CONFIG_PATH}')
else:
    print(f'✅ Config found!')

config      = load_config(str(CONFIG_PATH))

# ── Set reproducibility seed ─────────────────────────────────────────────
set_seed(config['training']['seed'])

# ── Device ───────────────────────────────────────────────────────────────
device = get_device(config)

print('\n📋 Key Configuration:')
print(f"   Model:       {config['model']['name']}")
print(f"   Image size:  {config['dataset']['image_size']}")
print(f"   Batch size:  {config['training']['batch_size']}")
print(f"   Epochs:      {config['training']['epochs']}")
print(f"   LR:          {config['training']['learning_rate']}")
print(f"   AMP:         {config['training']['amp']}")

🔍 Looking for config at: /content/config/config.yaml
   File exists: True
✅ Config found!

📋 Key Configuration:
   Model:       RT-DETR-L
   Image size:  640
   Batch size:  8
   Epochs:      150
   LR:          0.0001
   AMP:         True


In [4]:
# ── [Optional] Override config values for quick experiments ──────────────
# Uncomment and modify as needed:

# config['training']['epochs']     = 50       # Quick test run
# config['training']['batch_size'] = 4        # If GPU VRAM is limited
# config['training']['learning_rate'] = 5e-5  # Lower LR for fine-tuning
# config['dataset']['image_size'] = 416       # Smaller input (faster)

print('Config ready. Modify the lines above to experiment.')

Config ready. Modify the lines above to experiment.


---
## 3 — Dataset Preparation

**Expected folder structure before running this section:**
```
data/
  raw/
    images/   ← your olive images (.jpg / .png)
    labels/   ← YOLO format .txt files (one per image)
```

**YOLO label format** (one line per olive):
```
0  x_center  y_center  width  height
```
All values normalized 0–1. Class ID for olive = 0.

**Tip:** Use [Roboflow](https://roboflow.com) or [LabelImg](https://github.com/HumanSignal/labelImg) to annotate your images.

In [13]:
# ── Split dataset into train / val / test ────────────────────────────
from data.dataset import split_dataset, generate_data_yaml, build_dataloaders

# Split (only needed once; skip if already split)
RAW_IMAGES = PROJECT_ROOT / 'data' / 'raw' / 'images'
if RAW_IMAGES.exists() and any(RAW_IMAGES.iterdir()):
    split_dataset(config)
else:
    print('⚠️  data/raw/images is empty. Add your olive images first!')
    print('   Skipping split for now — using dummy structure.')

    # Create dummy structure for notebook to continue
    for split in ['train', 'val', 'test']:
        for sub in ['images', 'labels']:
            Path(f'data/processed/{split}/{sub}').mkdir(parents=True, exist_ok=True)

⚠️  data/raw/images is empty. Add your olive images first!
   Skipping split for now — using dummy structure.


In [14]:
# ── Debug: Check path resolution ─────────────────────────────────────────
print(f'🔍 PROJECT_ROOT: {PROJECT_ROOT}')

raw_path = PROJECT_ROOT / 'data' / 'raw' / 'images'
print(f'🔍 RAW_IMAGES path: {raw_path}')
print(f'   Exists: {raw_path.exists()}')

if raw_path.exists():
    files = list(raw_path.iterdir())
    print(f'   Files found: {len(files)}')
else:
    # Try alternative path
    alt_path = Path('data/raw/images')
    print(f'🔍 Trying alternative: {alt_path}')
    print(f'   Resolved to: {alt_path.resolve()}')
    print(f'   Exists: {alt_path.exists()}')
    if alt_path.exists():
        files = list(alt_path.iterdir())
        print(f'   Files found: {len(files)}')

🔍 PROJECT_ROOT: /content
🔍 RAW_IMAGES path: /content/data/raw/images
   Exists: False
🔍 Trying alternative: data/raw/images
   Resolved to: /content/data/raw/images
   Exists: False


In [9]:
# ── Generate data.yaml for Ultralytics ───────────────────────────────────
data_yaml_path = generate_data_yaml(config, 'data/data.yaml')
print(f'\ndata.yaml contents:')
with open(data_yaml_path) as f:
    print(f.read())


data.yaml contents:
names:
- olive
nc: 1
path: data
test: data/processed/test/images
train: data/processed/train/images
val: data/processed/val/images



In [10]:
# ── Dataset statistics ───────────────────────────────────────────────────
def count_dataset_stats(processed_dir: str) -> dict:
    stats = {}
    for split in ['train', 'val', 'test']:
        img_dir = Path(processed_dir) / split / 'images'
        lbl_dir = Path(processed_dir) / split / 'labels'
        n_imgs  = len(list(img_dir.glob('*.*'))) if img_dir.exists() else 0
        
        total_olives = 0
        if lbl_dir.exists():
            for lf in lbl_dir.glob('*.txt'):
                lines = lf.read_text().strip().splitlines()
                total_olives += len([l for l in lines if l.strip()])
        
        stats[split] = {
            'images':  n_imgs,
            'olives':  total_olives,
            'avg_per_image': round(total_olives / max(n_imgs, 1), 2),
        }
    return stats

stats = count_dataset_stats('data/processed')
print('\n📊 Dataset Statistics:')
print(f"{'Split':<10} {'Images':>8} {'Olives':>10} {'Avg/Image':>12}")
print('-' * 44)
for split, s in stats.items():
    print(f"{split:<10} {s['images']:>8} {s['olives']:>10} {s['avg_per_image']:>12}")


📊 Dataset Statistics:
Split        Images     Olives    Avg/Image
--------------------------------------------
train             0          0          0.0
val               0          0          0.0
test              0          0          0.0


---
## 4 — Exploratory Data Analysis

In [ ]:
# ── Visualize sample images with annotations ─────────────────────────────
def show_sample_annotations(images_dir: str, labels_dir: str, n: int = 6):
    img_dir = Path(images_dir)
    lbl_dir = Path(labels_dir)
    files   = sorted(img_dir.glob('*.jpg'))[:n] + sorted(img_dir.glob('*.png'))[:n]
    files   = files[:n]
    
    if not files:
        print('No images found.')
        return
    
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    fig.suptitle('Sample Annotations', fontsize=14, fontweight='bold')
    
    for ax, img_path in zip(axes.flatten(), files):
        img = cv2.imread(str(img_path))
        if img is None:
            ax.text(0.5, 0.5, f'Failed to load\n{img_path.name}', 
                    ha='center', va='center', transform=ax.transAxes)
            ax.axis('off')
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        
        lbl_path = lbl_dir / (img_path.stem + '.txt')
        count = 0
        if lbl_path.exists():
            for line in lbl_path.read_text().strip().splitlines():
                parts = line.split()
                if len(parts) == 5:
                    xc, yc, bw, bh = map(float, parts[1:])
                    x1 = int((xc - bw/2) * w)
                    y1 = int((yc - bh/2) * h)
                    bw_ = int(bw * w)
                    bh_ = int(bh * h)
                    rect = patches.Rectangle(
                        (x1, y1), bw_, bh_,
                        linewidth=1.5, edgecolor='#00C853', facecolor='none'
                    )
                    ax.add_patch(rect)
                    count += 1
        
        ax.imshow(img)
        ax.set_title(f'{img_path.name} | {count} olives', fontsize=9)
        ax.axis('off')
    
    plt.tight_layout()
    plt.savefig('logs/sample_annotations.png', dpi=120, bbox_inches='tight')
    plt.show()

Path('logs').mkdir(exist_ok=True)
show_sample_annotations('data/processed/train/images', 'data/processed/train/labels')

In [ ]:
# ── Bounding box size distribution ───────────────────────────────────────
def analyze_box_distribution(labels_dir: str):
    widths, heights, counts = [], [], []
    lbl_dir = Path(labels_dir)
    
    for lf in lbl_dir.glob('*.txt'):
        lines = [l for l in lf.read_text().strip().splitlines() if l.strip()]
        counts.append(len(lines))
        for line in lines:
            parts = line.split()
            if len(parts) == 5:
                widths.append(float(parts[3]) * 100)   # % of image width
                heights.append(float(parts[4]) * 100)  # % of image height
    
    if not widths:
        print('No labels found for analysis.')
        return
    
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    fig.suptitle('Label Distribution Analysis', fontsize=13, fontweight='bold')
    
    axes[0].hist(widths,  bins=30, color='#4CAF50', alpha=0.8, edgecolor='white')
    axes[0].set_title('Box Width Distribution (%)')
    axes[0].set_xlabel('Width (% of image)')
    axes[0].set_ylabel('Count')
    
    axes[1].hist(heights, bins=30, color='#2196F3', alpha=0.8, edgecolor='white')
    axes[1].set_title('Box Height Distribution (%)')
    axes[1].set_xlabel('Height (% of image)')
    
    axes[2].hist(counts, bins=range(0, max(counts)+2), color='#FF9800', alpha=0.8, edgecolor='white')
    axes[2].set_title('Olive Count per Image')
    axes[2].set_xlabel('# Olives')
    
    print(f'📐 Box size stats:')
    print(f'   Width  — mean: {np.mean(widths):.1f}%, median: {np.median(widths):.1f}%')
    print(f'   Height — mean: {np.mean(heights):.1f}%, median: {np.median(heights):.1f}%')
    print(f'   Olives — mean: {np.mean(counts):.1f}/img, max: {max(counts)}/img')
    
    plt.tight_layout()
    plt.savefig('logs/box_distribution.png', dpi=120, bbox_inches='tight')
    plt.show()

analyze_box_distribution('data/processed/train/labels')

---
## 5 — Model Architecture Overview

In [ ]:
# ── Architecture diagram (text) ───────────────────────────────────────────
arch_diagram = '''
┌─────────────────────────────────────────────────────────────────────┐
│                    OliveVision Architecture                         │
│                  (RT-DETR-L + Custom Enhancements)                  │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│  Input Image (640×640×3)                                            │
│       │                                                             │
│       ▼                                                             │
│  ┌──────────────────────────────────┐                               │
│  │  ResNet-101-D Backbone           │  ← COCO pretrained            │
│  │  (Deformable conv stem)          │                               │
│  │   C3: 80×80×512                  │                               │
│  │   C4: 40×40×1024                 │                               │
│  │   C5: 20×20×2048                 │                               │
│  └────────────┬─────────────────────┘                               │
│               │                                                     │
│               ▼                                                     │
│  ┌──────────────────────────────────┐                               │
│  │  BiFPN Neck (3 layers, d=256)    │  ← Bi-directional FPN        │
│  │  Weighted feature fusion         │                               │
│  │   P3: 80×80×256                  │                               │
│  │   P4: 40×40×256                  │                               │
│  │   P5: 20×20×256                  │                               │
│  └────────────┬─────────────────────┘                               │
│               │                                                     │
│               ▼                                                     │
│  ┌──────────────────────────────────┐                               │
│  │  CBAM (per level)                │  ← Channel + Spatial Attn    │
│  │  Channel attention (SE-style)    │                               │
│  │  Spatial attention (7×7 conv)    │                               │
│  └────────────┬─────────────────────┘                               │
│               │                                                     │
│               ▼  (on P5 only)                                       │
│  ┌──────────────────────────────────┐                               │
│  │  AIFI Encoder                    │  ← Global context on P5       │
│  │  Multi-head self-attention       │                               │
│  │  Intra-scale feature interaction │                               │
│  └────────────┬─────────────────────┘                               │
│               │  Flatten + Concat all levels                        │
│               ▼  Memory: (B, ΣHW, 256)                              │
│  ┌──────────────────────────────────┐                               │
│  │  RT-DETR Decoder (6 layers)      │  ← 300 object queries        │
│  │  Self-attention on queries       │                               │
│  │  Cross-attention with memory     │                               │
│  │  FFN (feed-forward network)      │                               │
│  └────────────┬─────────────────────┘                               │
│               │                                                     │
│               ▼                                                     │
│  ┌──────────────────────────────────┐                               │
│  │  IoU-Aware Head                  │                               │
│  │  cls_logits  (B, 300, 1)         │                               │
│  │  boxes       (B, 300, 4)         │  [xc, yc, w, h] normalized   │
│  │  iou_pred    (B, 300, 1)         │                               │
│  └────────────┬─────────────────────┘                               │
│               │                                                     │
│               ▼                                                     │
│  ┌──────────────────────────────────┐                               │
│  │  Hungarian Matching + Loss       │  ← Training only              │
│  │  Focal-CE + L1 + GIoU + IoU-aw  │                               │
│  └──────────────────────────────────┘                               │
│                                                                     │
│  Inference: conf filter → (optional NMS) → draw → count            │
└─────────────────────────────────────────────────────────────────────┘
'''
print(arch_diagram)

In [ ]:
# ── Load model and print parameter count ─────────────────────────────────
from ultralytics import RTDETR

print('🔄 Loading RT-DETR-L (downloading pretrained weights if needed)...')
model = RTDETR('rtdetr-l.pt')   # Downloads COCO-pretrained weights

total_params    = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'\n🧠 Model Summary:')
print(f'   Total parameters:     {total_params:>12,}')
print(f'   Trainable parameters: {trainable_params:>12,}')
print(f'   Model size:           {total_params * 4 / 1e6:>10.1f} MB')

---
## 6 — Training

Using **Ultralytics** training pipeline for RT-DETR which handles:
- Automatic mixed precision (AMP)
- Learning rate scheduling with warmup
- Augmentation (mosaic, mixup, copy-paste, etc.)
- Checkpointing (best + last)
- Metric logging
- EarlyStopping

In [ ]:
# ── Training configuration ──────────────────────────────────────────────────
TRAIN_ARGS = dict(
    data      = 'data/data.yaml',          # Dataset config
    epochs    = config['training']['epochs'],
    imgsz     = config['dataset']['image_size'],
    batch     = config['training']['batch_size'],
    workers   = config['training']['num_workers'],
    device    = 0 if torch.cuda.is_available() else 'cpu',
    
    # Optimizer
    optimizer = config['training']['optimizer'],
    lr0       = config['training']['learning_rate'],
    weight_decay = config['training']['weight_decay'],
    
    # Scheduler
    warmup_epochs = config['training']['warmup_epochs'],
    cos_lr        = True,
    
    # Loss weights
    cls = config['training']['loss']['cls_weight'],
    box = config['training']['loss']['bbox_weight'],
    
    # Augmentation
    mosaic     = config['augmentation']['train']['mosaic'],
    mixup      = config['augmentation']['train']['mixup'],
    copy_paste = config['augmentation']['train']['copy_paste'],
    flipud     = config['augmentation']['train']['vertical_flip'],
    fliplr     = config['augmentation']['train']['horizontal_flip'],
    degrees    = config['augmentation']['train']['random_rotation'],
    hsv_h      = config['augmentation']['train']['hsv_h'],
    hsv_s      = config['augmentation']['train']['hsv_s'],
    hsv_v      = config['augmentation']['train']['hsv_v'],
    
    # Checkpointing
    project    = 'models',
    name       = 'olive_rtdetr',
    save       = True,
    save_period = config['training']['save_every'],
    
    # Misc
    amp        = config['training']['amp'],
    patience   = config['training']['early_stopping']['patience'],
    verbose    = True,
    plots      = True,
)

print('🎯 Training Configuration:')
for k, v in TRAIN_ARGS.items():
    print(f'   {k:<20}: {v}')

In [ ]:
print('🚀 Starting training...')
print(f'   Estimated time: ~{int(TRAIN_ARGS["epochs"]) * 0.8:.0f} minutes on RTX 3090')

results = model.train(**TRAIN_ARGS)

print('\n✅ Training complete!')
print(f'   Best model: models/olive_rtdetr/weights/best.pt')
print(f'   Last model: models/olive_rtdetr/weights/last.pt')

In [ ]:
# ── Copy best weights to standard location ────────────────────────────────
import shutil

src  = Path('models/olive_rtdetr/weights/best.pt')
dst  = Path('models/checkpoints/best.pt')
dst.parent.mkdir(parents=True, exist_ok=True)

if src.exists():
    shutil.copy(src, dst)
    print(f'✅ Best weights copied → {dst}')
else:
    print(f'⚠️  best.pt not found at {src}. Training may not have completed.')

---
## 7 — Evaluation

In [ ]:
# ── Run validation on best model ─────────────────────────────────────────
BEST_WEIGHTS = 'models/checkpoints/best.pt'

if Path(BEST_WEIGHTS).exists():
    best_model = RTDETR(BEST_WEIGHTS)
    val_results = best_model.val(
        data    = 'data/data.yaml',
        split   = 'val',
        imgsz   = config['dataset']['image_size'],
        batch   = config['training']['batch_size'],
        conf    = config['evaluation']['conf_threshold'],
        iou     = config['evaluation']['iou_threshold'],
        device  = 0 if torch.cuda.is_available() else 'cpu',
        verbose = True,
    )
    
    print('\n📊 Validation Metrics:')
    print(f'   mAP@50:    {val_results.box.map50:.4f}')
    print(f'   mAP@50-95: {val_results.box.map:.4f}')
    print(f'   Precision: {val_results.box.mp:.4f}')
    print(f'   Recall:    {val_results.box.mr:.4f}')
else:
    print(f'⚠️  No weights found at {BEST_WEIGHTS}. Train the model first.')

In [ ]:
# ── Test set evaluation ───────────────────────────────────────────────────
if Path(BEST_WEIGHTS).exists():
    test_results = best_model.val(
        data    = 'data/data.yaml',
        split   = 'test',
        imgsz   = config['dataset']['image_size'],
        batch   = config['training']['batch_size'],
        conf    = config['evaluation']['conf_threshold'],
        iou     = config['evaluation']['iou_threshold'],
        device  = 0 if torch.cuda.is_available() else 'cpu',
        verbose = True,
    )

    print('\n🧪 Test Set Metrics:')
    print(f'   mAP@50:    {test_results.box.map50:.4f}')
    print(f'   mAP@50-95: {test_results.box.map:.4f}')
    print(f'   Precision: {test_results.box.mp:.4f}')
    print(f'   Recall:    {test_results.box.mr:.4f}')

In [ ]:
# ── Count accuracy (MAE / RMSE) on test set ───────────────────────────────
import glob

def evaluate_counting(model, test_images_dir: str, test_labels_dir: str,
                       conf: float = 0.35, iou: float = 0.45,
                       imgsz: int = 640, device='cpu') -> dict:
    """
    Evaluate olive counting accuracy.
    Returns MAE, RMSE, and per-image count comparison.
    """
    img_dir = Path(test_images_dir)
    lbl_dir = Path(test_labels_dir)
    
    image_files = sorted(list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png')))
    if not image_files:
        print('No test images found.')
        return {}
    
    errors  = []
    records = []
    
    for img_path in tqdm(image_files, desc='Counting evaluation'):
        lbl_path = lbl_dir / (img_path.stem + '.txt')
        gt_count = 0
        if lbl_path.exists():
            lines    = [l for l in lbl_path.read_text().strip().splitlines() if l.strip()]
            gt_count = len(lines)
        
        results   = model.predict(
            str(img_path), conf=conf, iou=iou, imgsz=imgsz,
            device=device, verbose=False
        )
        pred_count = len(results[0].boxes) if results[0].boxes else 0
        err = abs(pred_count - gt_count)
        errors.append(err)
        records.append({'image': img_path.name, 'gt': gt_count, 'pred': pred_count, 'error': err})
    
    mae  = np.mean(errors)
    rmse = math.sqrt(np.mean(np.array(errors)**2))
    acc  = np.mean([1 if e == 0 else 0 for e in errors]) * 100
    
    return {'MAE': mae, 'RMSE': rmse, 'Exact Accuracy %': acc, 'records': records}


if Path(BEST_WEIGHTS).exists():
    device_str = '0' if torch.cuda.is_available() else 'cpu'
    count_metrics = evaluate_counting(
        best_model,
        'data/processed/test/images',
        'data/processed/test/labels',
        conf=config['evaluation']['conf_threshold'],
        iou=config['evaluation']['iou_threshold'],
        imgsz=config['dataset']['image_size'],
        device=device_str,
    )
    
    if count_metrics:
        print('\n🫒 Olive Counting Accuracy:')
        print(f"   MAE:             {count_metrics['MAE']:.3f}  (mean absolute error in olive count)")
        print(f"   RMSE:            {count_metrics['RMSE']:.3f}")
        print(f"   Exact Accuracy:  {count_metrics['Exact Accuracy %']:.1f}%  (images with exact count)")

In [ ]:
# ── Plot training curves from Ultralytics results.csv ────────────────────
results_csv = Path('models/olive_rtdetr/results.csv')

if results_csv.exists():
    import pandas as pd
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle('OliveVision Training Curves', fontsize=15, fontweight='bold')
    
    plots = [
        ('train/box_loss',  'Train Box Loss',   'b'),
        ('val/box_loss',    'Val Box Loss',     'r'),
        ('train/cls_loss',  'Train Cls Loss',   'b'),
        ('val/cls_loss',    'Val Cls Loss',     'r'),
        ('metrics/mAP50',   'mAP@50',           'g'),
        ('metrics/mAP50-95','mAP@50-95',        'm'),
    ]
    
    for ax, (col, title, color) in zip(axes.flatten(), plots):
        if col in df.columns:
            ax.plot(df['epoch'], df[col], color=color, linewidth=2)
            ax.set_title(title, fontweight='bold')
            ax.set_xlabel('Epoch')
            ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('logs/training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('📊 Training curves saved → logs/training_curves.png')
else:
    print('⚠️  results.csv not found. Train the model first.')

In [ ]:
# ── Visualize predictions on test images ────────────────────────────────
def visualize_predictions(model, images_dir: str, n: int = 6,
                           conf: float = 0.35, iou: float = 0.45, imgsz: int = 640):
    img_dir = Path(images_dir)
    files   = sorted(img_dir.glob('*.jpg'))[:n] + sorted(img_dir.glob('*.png'))[:n]
    files   = files[:n]
    
    if not files:
        print('No test images found.')
        return
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle('Test Set Predictions', fontsize=13, fontweight='bold')
    
    for ax, img_path in zip(axes.flatten(), files):
        results = model.predict(
            str(img_path), conf=conf, iou=iou, imgsz=imgsz,
            device='0' if torch.cuda.is_available() else 'cpu', verbose=False
        )
        
        img  = cv2.imread(str(img_path))
        if img is None:
            ax.text(0.5, 0.5, f'Failed to load\n{img_path.name}', 
                    ha='center', va='center', transform=ax.transAxes)
            ax.axis('off')
            continue
        img  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        
        count = 0
        if results[0].boxes:
            # results[0].boxes.xyxy is already numpy, no need for .cpu()
            boxes = results[0].boxes.xyxy if isinstance(results[0].boxes.xyxy, np.ndarray) else results[0].boxes.xyxy.cpu().numpy()
            for box in boxes:
                x1, y1, x2, y2 = box.astype(int)
                rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                          linewidth=2, edgecolor='#00E676', facecolor='none')
                ax.add_patch(rect)
                count += 1
        
        ax.imshow(img)
        ax.set_title(f'{img_path.name}\nDetected: {count} olives', fontsize=9)
        ax.axis('off')
    
    plt.tight_layout()
    plt.savefig('logs/test_predictions.png', dpi=120, bbox_inches='tight')
    plt.show()


if Path(BEST_WEIGHTS).exists():
    visualize_predictions(best_model, 'data/processed/test/images',
                          conf=config['evaluation']['conf_threshold'],
                          iou=config['evaluation']['iou_threshold'],
                          imgsz=config['dataset']['image_size'])

---
## 8 — Export & Deployment

In [ ]:
# ── Export to ONNX (for deployment on edge / web) ────────────────────────
if Path(BEST_WEIGHTS).exists():
    print('🔄 Exporting to ONNX...')
    onnx_path = best_model.export(
        format='onnx',
        imgsz=config['dataset']['image_size'],
        half=True if torch.cuda.is_available() else False,
        simplify=True,
        dynamic=False,
    )
    print(f'✅ ONNX model saved → {onnx_path}')
    
    # Optional: TorchScript
    # ts_path = best_model.export(format='torchscript')
    # print(f'✅ TorchScript saved → {ts_path}')
    
    # Optional: TensorRT (NVIDIA only)
    # trt_path = best_model.export(format='engine', half=True)
    # print(f'✅ TensorRT saved → {trt_path}')

In [ ]:
# ── Benchmark inference speed ─────────────────────────────────────────────
if Path(BEST_WEIGHTS).exists():
    import time
    
    device_str = '0' if torch.cuda.is_available() else 'cpu'
    dummy_img  = np.random.randint(0, 255, (640, 640, 3), dtype=np.uint8)
    
    # Warmup
    for _ in range(5):
        best_model.predict(dummy_img, device=device_str, verbose=False)
    
    # Benchmark (50 iterations)
    times = []
    for _ in tqdm(range(50), desc='Benchmarking'):
        t0 = time.perf_counter()
        best_model.predict(dummy_img, device=device_str, verbose=False)
        times.append(time.perf_counter() - t0)
    
    avg_ms = np.mean(times) * 1000
    fps    = 1 / np.mean(times)
    
    print(f'\n⚡ Inference Benchmark (640×640):')
    print(f'   Average latency: {avg_ms:.1f} ms')
    print(f'   FPS:             {fps:.1f}')
    print(f'   Device:          {"GPU" if torch.cuda.is_available() else "CPU"}')

---
## 9 — Quick Inference Test

In [ ]:
# ── Test on a single image ───────────────────────────────────────────────
# Change this path to any olive image on your system
TEST_IMAGE = 'data/processed/test/images/<your_image>.jpg'

if Path(TEST_IMAGE).exists() and Path(BEST_WEIGHTS).exists():
    results = best_model.predict(
        TEST_IMAGE,
        conf=config['inference']['conf_threshold'],
        iou=config['inference']['iou_threshold'],
        imgsz=config['dataset']['image_size'],
        device='0' if torch.cuda.is_available() else 'cpu',
        verbose=False,
    )
    
    count = len(results[0].boxes) if results[0].boxes else 0
    
    img = cv2.imread(TEST_IMAGE)
    if img is None:
        print(f'Failed to load image: {TEST_IMAGE}')
    else:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        fig, ax = plt.subplots(1, 1, figsize=(12, 8))
        ax.imshow(img)
        
        if results[0].boxes:
            # results[0].boxes.xyxy is already numpy, no need for .cpu()
            boxes = results[0].boxes.xyxy if isinstance(results[0].boxes.xyxy, np.ndarray) else results[0].boxes.xyxy.cpu().numpy()
            for box in boxes:
                x1, y1, x2, y2 = box.astype(int)
                rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                           linewidth=2, edgecolor='#00E676', facecolor='none')
                ax.add_patch(rect)
        
        ax.set_title(f'🫒 Detected Olives: {count}', fontsize=16, fontweight='bold', color='#00C853')
        ax.axis('off')
        plt.tight_layout()
        plt.show()
        print(f'Olive count: {count}')
else:
    print('Update TEST_IMAGE path above to a real image, and ensure best.pt exists.')

In [ ]:
# ── Launch real-time webcam inference ────────────────────────────────────
# ⚠️  This opens a CV2 window. Press 'q' to exit.

# Uncomment to run:
# import subprocess
# subprocess.run([
#     'python', 'inference/infer.py',
#     '--weights', BEST_WEIGHTS,
#     '--source', '0',         # 0 = webcam
#     '--config', CONFIG_PATH,
# ])

In [ ]:
# ── Final summary ─────────────────────────────────────────────────────────
print('='*60)
print('  OliveVision FYP — Summary')
print('='*60)
print(f'  Architecture:  RT-DETR-L + CBAM + BiFPN + IoU-Head')
print(f'  Best weights:  models/checkpoints/best.pt')
print(f'  ONNX export:   models/olive_rtdetr/weights/best.onnx')
print(f'  Training log:  models/olive_rtdetr/results.csv')
print(f'  Plots:         logs/')
print('='*60)
print()
print('  To run real-time detection:')
print('  $ python inference/infer.py --source 0')
print()
print('  To run on a video:')
print('  $ python inference/infer.py --source path/to/video.mp4')
print('='*60)